In [80]:
import pandas as pd

#DISPLAY FORMATTING
pd.set_option('display.float_format', "{:,.3f}".format)

filename = "FC26.csv"
df = pd.read_csv(filename)
df = df [["long_name","age","height_cm","overall", "potential", "player_positions", "value_eur", "release_clause_eur", "club_name","club_position","league_name", "league_level",
          "preferred_foot","weak_foot","skill_moves","international_reputation","work_rate","body_type"]]


C:\Users\Salman\AppData\Local\Temp\ipykernel_19968\1896167798.py:7: DtypeWarning: Columns (0: player_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filename)


In [81]:
df.drop_duplicates(inplace=True)
df.drop(columns=["work_rate"], inplace = True) # drop as it is empty for all
for i in df[df.release_clause_eur.isnull()].index:
    df.loc[i, "release_clause_eur"] = df.loc[i, "value_eur"] *1.25

c = 'player_positions' #variable to hold column name
df[c] = df[c].str.split(",").str[0].str.strip()
#now encode non numeric columns
df.drop(columns = "long_name", inplace=True)
nonNumeric = df.select_dtypes(exclude = ["number"])

In [82]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
for c in nonNumeric:
    df[c] = LabelEncoder().fit_transform(df[c])

#no column really requires scaling, lets still go for age
scaler = StandardScaler()
columns_to_scale = ['age']
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

In [83]:
#in our dataset, outliers should not be removed as they are main points, so I am not removing for most columns, removing only for league_level
df_new = df.copy()
c = "league_level"
Q1 = df_new[c].quantile(0.25)
Q3 = df_new[c].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5*IQR
higher_bound = Q3 + 1.5*IQR

df_new = df_new[(df_new[c] >= lower_bound) & (df_new[c] <=higher_bound)]

In [84]:
#oversampling,undersampling and combination
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek
X = df_new.drop(columns = ["value_eur"])
y = df_new["value_eur"] #target column




In [85]:
##FEATURE TRANSFORMATION
import numpy as np
#my distribution looks like a gamma distribution 
#as my target column is not 0/1, I cannot simply apply undersampling, oversampling or combination.
#i am applying log trasnform beacuse most of the players have low market values with very few having huge values
#it will lessen the influence of extremely valuable players, and create a more stable relationship between features and target values.
#so the model may not spend too much effort fitting a few elite players and ignore ordinary players.


df_new = pd.concat([X,y], axis = 1)
df_new['log_value'] = np.log1p(df_new['value_eur'])
df_new['log_rc'] = np.log1p(df_new['release_clause_eur'])

#altho overall and potential do give similiar info and have similar trend, but I cannot drop them because they still give different information

unnecessary_cols = ["player_positions","club_name","height_cm","preferred_foot", "value_eur", "release_clause_eur"]
df_new = df_new.drop(columns = unnecessary_cols)




In [86]:
#2.Variance Threshold
#works on numeric cols only, so lets separate
X = df_new.drop(columns = ["log_value"])
y = df_new["log_value"]

numeric_X = X.select_dtypes(exclude=["str"])


In [87]:
#as international repo, preferred foot variate quite less but stil not around 0.01 or very low, so right now we cannot make a
#rigid decision to directly drop them, maybe they have some influence in predicting market values, we'll see with our feature selection techniques
#applying Variance Threshold, to automatically remove columns with var <0.01, THIS WOULD REMOVE NOTHING OF COURSE FOR NOW
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(numeric_X)

selected_columns = numeric_X.columns[selector.get_support()]
removed_columns = numeric_X.columns[~selector.get_support()]


In [88]:
#3.Mutual Info
from sklearn.feature_selection import mutual_info_regression

mi_scores = mutual_info_regression(X,y,random_state = 42)
mi_dframe = pd.DataFrame({"Feature":X.columns, "Scores":mi_scores})

mi_dframe = mi_dframe.sort_values(by= "Scores", ascending = False)

#REMOVE COLUMNS IN NEXT STEP - SELECT KBEST

In [89]:
#4.SelectKBest
#we have chi2, f_classif, f_regession, mutual_regression

#both regression ones can be used but f_regression is for more linear relatinoshisps,
#mutual_regression can help us better I think because it can detect no linear rship too
#  because age 18 player has less value, a mature 25 player has more, then 30 yo player has less value
#so it is not a straitht line realtionsihp


#REMEMBER X must be encoded as categorical variables should be encoded
target = ["log_value"]
X = df_new[selected_columns]
y = df_new[target]

from sklearn.feature_selection import SelectKBest

selector = SelectKBest (score_func=mutual_info_regression, k = 'all')
selector.fit(X,y)
#now select till age, k = 5
kVal = 5
selector = SelectKBest(score_func=mutual_info_regression, k = kVal)
X_selected = selector.fit_transform(X,y)

selected_columns = X.columns[selector.get_support()]
removed_columns = X.columns[~selector.get_support()]



C:\Users\Salman\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\Salman\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [90]:
import math
df_final = pd.concat([df_new[selected_columns], y], axis = 1)
df_final.drop(columns = ["league_name"],inplace = True)

print(df_final.columns)


Index(['age', 'overall', 'potential', 'log_rc', 'log_value'], dtype='str')


***Model Training***

In [91]:
def format_value(val):
    if val >=1_000_000:
        return f"{val/1_000_000:.2f}M"
    elif val >=1_000:
        return f"{val/1_000:.0f}K"
    else: return str(val)


In [92]:
from sklearn.model_selection import train_test_split

#here I have dropped release clause as well as it can become a biased indicator
X = df_final.drop(columns = ["log_value", "log_rc"])
y = df_final["log_value"]


x_train, x_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [93]:
from sklearn.linear_model import LinearRegression
#to be used in comparison tables of all models
actual_values  = np.exp(y_test)-1 #convert back to Euros
actual_values_display = pd.Series(np.vectorize(format_value)(actual_values), index = x_test.index, name = "Actual Value")

#train model
model = LinearRegression()
model.fit(x_train,y_train)
y_pred_lr = model.predict(x_test)

#convert predictions to euros and display resulsts
lr_predicted_values = np.exp(y_pred_lr)-1
lr_error = lr_predicted_values - actual_values
lr_results = pd.concat([
    x_test,
    actual_values_display,
    pd.Series(np.vectorize(format_value)(lr_predicted_values), index=x_test.index, name="Predicted Value"),
    pd.Series(np.vectorize(format_value)(np.abs(lr_error)), index=x_test.index, name="Prediction Error")
], axis=1)
print(lr_results.head())




         age  overall  potential Actual Value Predicted Value  \
10126  0.582       65         65         725K            674K   
12233  0.582       66         67         800K            801K   
13003  0.582       62         62         400K            373K   
7690  -0.675       68         78        2.70M           2.44M   
14014 -0.885       65         75        1.50M           1.58M   

        Prediction Error  
10126                51K  
12233  653.8210675722221  
13003                27K  
7690                256K  
14014                84K  


In [94]:
#Evaluation Metrics
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error,accuracy_score,precision_score,recall_score,f1_score
print("For Linear Regression Model, metrics are:")
print("R2 score is: ", r2_score(actual_values,lr_predicted_values))
print("Mean Absolute Error: ", mean_absolute_error(actual_values,lr_predicted_values))
print("Mean Squared Error is: ", mean_squared_error(actual_values,lr_predicted_values))
print("Root Mean Squared Error is: ", np.sqrt(mean_squared_error(actual_values,lr_predicted_values)))

For Linear Regression Model, metrics are:
R2 score is:  0.9056242082286446
Mean Absolute Error:  684196.6773584642
Mean Squared Error is:  4855872380724.461
Root Mean Squared Error is:  2203604.4065858237
